# 04. Iterators, generators and context managers

## Learning goals
1. Implement the iterator protocol by hand.
2. Write generators, use `send` and `yield from`.
3. Compare memory use of lazy and eager pipelines.
4. Write context managers with a class and with `contextlib`.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## The iterator protocol
`iter(obj)` calls `__iter__`. A `for` loop calls `__next__` until `StopIteration`.

In [ ]:
from understanding_python.functional import Countdown
c = Countdown(3)
assert iter(c) is c
assert [next(c), next(c), next(c)] == [3, 2, 1]
try:
    next(c)
    raise AssertionError("expected StopIteration")
except StopIteration:
    pass
assert list(Countdown(4)) == [4, 3, 2, 1]
it = iter([1, 2]); assert list(it) == [1, 2] and list(it) == []   # exhausted

## Generators
A function with `yield` returns a generator. It pauses at each `yield` and keeps its local state.
Its frame costs O(1) memory no matter how many values it yields.

In [ ]:
from understanding_python.functional import fib_gen, running_mean
g = fib_gen(100)
assert next(g) == 0 and next(g) == 1
assert list(fib_gen(50)) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
m = running_mean()
next(m)
assert [m.send(v) for v in (10, 20, 60)] == [10, 15, 30]
def chain(*its):
    for it in its:
        yield from it
assert list(chain([1], (2, 3), range(4, 6))) == [1, 2, 3, 4, 5]

## Lazy against eager: memory
A list of n integers holds n references. A generator holds one frame.

In [ ]:
import sys
from understanding_python.bench import best_time
n = 50_000
eager = [i * i for i in range(n)]
lazy = (i * i for i in range(n))
print("list", sys.getsizeof(eager), "bytes; generator", sys.getsizeof(lazy), "bytes")
assert sys.getsizeof(lazy) < sys.getsizeof(eager) // 100
assert sum(lazy) == sum(eager)
first = next(i for i in range(10**9) if i * i > 500)   # stops early
assert first == 23

## itertools
`itertools` builds pipelines that stay lazy.

In [ ]:
import itertools as it
assert list(it.islice(it.count(5), 3)) == [5, 6, 7]
assert list(it.chain.from_iterable([[1], [2, 3]])) == [1, 2, 3]
assert list(it.accumulate([1, 2, 3, 4])) == [1, 3, 6, 10]
assert list(it.combinations("abc", 2)) == [("a", "b"), ("a", "c"), ("b", "c")]
assert [(k, len(list(v))) for k, v in it.groupby("aabbbc")] == [("a", 2), ("b", 3), ("c", 1)]
assert len(list(it.permutations(range(4)))) == 24

## Context managers
`with` calls `__enter__`, runs the body, then calls `__exit__` even when the body raises.
Returning True from `__exit__` suppresses the exception.

In [ ]:
from understanding_python.functional import Suppress, timer
import contextlib
with Suppress(ZeroDivisionError):
    1 / 0
try:
    with Suppress(KeyError):
        raise TypeError("not suppressed")
except TypeError:
    pass
out = {}
with timer(out):
    sum(range(1000))
assert out["elapsed"] >= 0
@contextlib.contextmanager
def opened(log):
    log.append("open")
    try:
        yield log
    finally:
        log.append("close")
events = []
try:
    with opened(events):
        raise RuntimeError
except RuntimeError:
    pass
assert events == ["open", "close"]

## Complexity: generator pipeline
Chain k generator stages over n items. Each item passes through each stage once, so the time is O(k n) and the memory is O(k) frames.
An eager pipeline needs the same time but O(n) memory per stage.

In [ ]:
def stage(src, f):
    for x in src:
        yield f(x)
def pipeline(n, k):
    s = iter(range(n))
    for _ in range(k):
        s = stage(s, lambda v: v + 1)
    return sum(s)
n, k = 2000, 5
assert pipeline(n, k) == sum(range(n)) + n * k
t1, t2 = best_time(lambda: pipeline(4000, 5), 3), best_time(lambda: pipeline(8000, 5), 3)
print(f"n=4000 {t1:.4f}s  n=8000 {t2:.4f}s  ratio {t2/t1:.2f} (linear predicts about 2)")
assert 1.0 < t2 / t1 < 4.0